# Smoke test for `ehtim.plotting.interactive`

Open in VSCode, pick the **jax-ehtim** kernel (top-right corner of the notebook), and run cells one at a time. Plotly figures render inline in the cell output — interactive (hover, zoom, pan, double-click to reset) without writing any HTML to disk.

In [1]:
import ehtim as eh
from ehtim.plotting import interactive

obs = eh.obsdata.load_uvfits('../data/sample.uvfits')
sites = sorted({s for pair in zip(obs.data['t1'], obs.data['t2']) for s in pair})
print('sites in sample.uvfits:', sites)

Welcome to eht-imaging! v 1.3.0 

Loading uvfits:  ../data/sample.uvfits
POLREP_UVFITS: circ
Number of uvfits Correlation Products: 4
No NX table in uvfits!
sites in sample.uvfits: ['ALMA', 'LMT', 'PDB', 'PV', 'SMA', 'SMT', 'SPT']


## `plot_bl` — single baseline

Pass `site1` and `site2` for one specific baseline. No legend (only one trace).

In [4]:
interactive.plot_bl(obs, 'ALMA', 'LMT', 'amp', show=False)

## `plot_bl` — all baselines, click-to-highlight

Omit `site1`/`site2` to plot every baseline as a uniform-gray trace. Then:

- **First click** on a legend entry → paints that baseline a distinct colour.
- **Second click** → hides it.
- **Third click** → back to gray default.
- **Double-click** → isolate (plotly default).

Use `interactive.display(fig)` instead of returning `fig` so the JS handler is wired up. `interactive.write_html(fig, 'out.html')` produces a standalone HTML with the same UX (good for sending Rohan).

In [5]:
fig = interactive.plot_bl(obs, field='amp', show=False)
interactive.display(fig)

Try other fields — `'phase'`, `'snr'`, `'uvdist'`. SNR-cut example: `interactive.plot_bl(obs, field='amp', snrcut=5, show=False)`.

## `plotall` — scatter any two visibility fields

Plotly counterpart of `obs.plotall(field1, field2, ...)`.

- **`tag_bl=True` (default):** one trace per baseline, toggleable via legend.
- **`tag_bl=False`:** all visibilities pooled into one trace — useful when you just want the cloud.

In [6]:
# UV coverage. conj=True overlays the Hermitian conjugate half (-u,-v).
# Click a baseline in the legend to highlight it in colour; click again to hide.
fig = interactive.plotall(obs, 'u', 'v', conj=True, show=False)
interactive.display(fig)

In [7]:
# Radplot: amplitude vs baseline length. Pooled mode — single trace, useful
# when you just want to see the overall distribution.
interactive.plotall(obs, 'uvdist', 'amp', tag_bl=False, show=False)

In [8]:
# Per-baseline radplot. Click in the legend to highlight one in colour.
fig = interactive.plotall(obs, 'uvdist', 'amp', tag_bl=True, show=False)
interactive.display(fig)

## `plot_gains` — needs a Caltable

`sample.uvfits` has no caltable attached, so we synthesise a tiny fake one. For a real demo, replace this with a Caltable from `self_cal(obs, im, caltable=True)`.

`pol='both'` draws R and L as separate traces per site — R uses circles, L uses squares, so they stay separable even within the same colour.

In [9]:
import numpy as np
from ehtim.caltable import Caltable

eht = eh.array.load_txt('../arrays/EHT2017.txt')
demo_sites = ['ALMA', 'LMT', 'SMA', 'SMT']
rng = np.random.default_rng(0)
times = np.linspace(0.0, 24.0, 20)
datadict = {
    s: np.array(
        [(t, 1.0 + 0.1*rng.standard_normal() + 0j,
              1.0 + 0.1*rng.standard_normal() + 0j) for t in times],
        dtype=[('time','f8'),('rscale','c16'),('lscale','c16')]
    ) for s in demo_sites
}
ct = Caltable(ra=0.0, dec=0.0, rf=230e9, bw=4e9, datadict=datadict,
              tarr=eht.tarr, source='demo', mjd=58000, timetype='GMST')

interactive.plot_gains(ct, sites='all', gain_type='amp', pol='both', show=False)

## `dashboard` — combined view

2x2 grid: image + amp-vs-uvdist + gains + D-terms. Use this to inspect a full reconstruction (or a model + synthetic obs + self-cal output) at a glance. Same legend semantics as the other plots — click to hide/show, double-click to isolate.

In [10]:
# Build a (Image, Obsdata, Caltable) triple end-to-end so the dashboard has
# real data to show. Takes ~10 s — self_cal does the heavy lifting.
from ehtim.calibrating import self_cal

im_dash = eh.image.load_txt('../models/avery_sgra_eofn.txt')
eht_dash = eh.array.load_txt('../arrays/EHT2017.txt')

# Synthesise small D-terms so the D-term panel isn't all-zero.
rng = np.random.default_rng(42)
n = len(eht_dash.tarr)
eht_dash.tarr['dr'] = rng.normal(0, 0.05, n) + 1j * rng.normal(0, 0.05, n)
eht_dash.tarr['dl'] = rng.normal(0, 0.05, n) + 1j * rng.normal(0, 0.05, n)

obs_dash = im_dash.observe(eht_dash, tint=30, tadv=600,
                            tstart=0, tstop=24, bw=4e9,
                            sgrscat=False, ampcal=False, phasecal=False)
ct_dash = self_cal.self_cal(obs_dash, im_dash, processes=-1,
                             ttype='direct', caltable=True)

fig_dash = interactive.dashboard(im_dash, obs_dash, ct_dash, show=False)
interactive.display(fig_dash)

Loading text image:  ../models/avery_sgra_eofn.txt
Generating empty observation file . . . 
Producing clean visibilities from image with nfft FT . . . 
Adding gain + phase errors to data and applying a priori calibration . . . 
   Applying gain corruption: ampcal-->False
   Applying atmospheric phase corruption: phasecal-->False
Adding thermal noise to data . . . 
No stations specified in self cal: defaulting to calibrating all stations!
Computing the Model Visibilities with direct Fourier Transform...
Producing clean visibilities from image with direct FT . . . 
Not Using Multiprocessing
Scan 106/107 : [----------------------------- ]99%
self_cal time: 7.962687 s
Producing clean visibilities from image with direct FT . . . 
